<a href="https://colab.research.google.com/github/azel2047/EDA2026/blob/main/Tugas%20Kelompok%20Pertemuan%203.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

# 1. Muat Data
path = '/content/drive/MyDrive/Explorasi Data Analis 2026 - Ugroseno Dwi Prakastyo/pertemuan3/Sales Transaction v.4a.csv'
df = pd.read_csv(path)

# 2. Preprocessing & Pembersihan Data
df['Date'] = pd.to_datetime(df['Date'])
df['YearMonth'] = df['Date'].dt.to_period('M')
df['IsCancelled'] = df['TransactionNo'].astype(str).str.startswith('C') | (df['Quantity'] < 0)
df['LineTotal'] = df['Price'] * df['Quantity']

df_clean = df[~df['IsCancelled']].copy()

print("=" * 60)
print(f"DATA BERHASIL DIMUAT: {len(df):,} baris total")
print("=" * 60)

# Pertanyaan 2: Top 10 Produk Berdasarkan Pendapatan
print("\n[PERTANYAAN 2] TOP 10 PRODUK DENGAN PENDAPATAN TERTINGGI:")
top_products = (
    df_clean.groupby(['ProductNo', 'ProductName'])
    .agg(TotalQty=('Quantity', 'sum'), TotalRevenue=('LineTotal', 'sum'))
    .sort_values(by='TotalRevenue', ascending=False)
    .reset_index()
)
print(top_products.head(10).to_string(index=False))

# Pertanyaan 3: Tren Penjualan Bulanan
print("\n" + "=" * 60)
print("[PERTANYAAN 3] TREN TRANSAKSI DAN PENDAPATAN PER BULAN:")
monthly_trend = (
    df_clean.groupby('YearMonth')
    .agg(
        TotalTransactions=('TransactionNo', 'nunique'),
        MonthlyRevenue=('LineTotal', 'sum'),
    )
    .reset_index()
)
print(monthly_trend.to_string(index=False))

# Pertanyaan 4: Kinerja Penjualan per Negara (Top 10)
print("\n" + "=" * 60)
print("[PERTANYAAN 4] TOP 10 NEGARA PENYUMBANG PENDAPATAN TERBESAR:")
country_perf = (
    df_clean.groupby('Country')
    .agg(
        ActiveCustomers=('CustomerNo', 'nunique'),
        TotalTransactions=('TransactionNo', 'nunique'),
        TotalRevenue=('LineTotal', 'sum'),
    )
    .sort_values(by='TotalRevenue', ascending=False)
    .reset_index()
)
print(country_perf.head(10).to_string(index=False))

# Pertanyaan 5: Analisis Pembatalan Pesanan
print("\n" + "=" * 60)
print("[PERTANYAAN 5] METRIK PEMBATALAN TRANSAKSI:")
total_trx = df['TransactionNo'].nunique()
cancelled_trx = df[df['IsCancelled']]['TransactionNo'].nunique()
cancellation_rate = (cancelled_trx / total_trx) * 100

print(f"- Total Transaksi Keseluruhan : {total_trx:,}")
print(f"- Total Transaksi Dibatalkan  : {cancelled_trx:,}")
print(f"- Rasio Pembatalan (Rate)     : {cancellation_rate:.2f}%\n")

print("Top 5 Produk yang Paling Sering Dibatalkan:")
top_cancelled = (
    df[df['IsCancelled']]
    .groupby(['ProductNo', 'ProductName'])
    .agg(
        CancelledQty=('Quantity', lambda x: abs(x.sum())),
        CancelFrequency=('TransactionNo', 'count'),
    )
    .sort_values(by='CancelledQty', ascending=False)
    .reset_index()
)
print(top_cancelled.head(5).to_string(index=False))

DATA BERHASIL DIMUAT: 536,350 baris total

[PERTANYAAN 2] TOP 10 PRODUK DENGAN PENDAPATAN TERTINGGI:
ProductNo                        ProductName  TotalQty  TotalRevenue
    23843          Paper Craft Little Birdie     80995    1002718.10
    23166     Medium Ceramic Top Storage Jar     78033     881990.18
    22197                     Popcorn Holder     56921     587433.94
    84077  World War 2 Gliders Asstd Designs     55047     569735.39
   85123A Cream Hanging Heart T-Light Holder     37956     484592.69
    84879      Assorted Colour Bird Ornament     36493     421318.74
    21212    Pack Of 72 Retrospot Cake Cases     36515     391485.03
    23084                 Rabbit Night Light     30788     329029.89
    22423           Regency Cakestand 3 Tier     13890     307483.85
   85099B            Jumbo Bag Red Retrospot     48478     297205.04

[PERTANYAAN 3] TREN TRANSAKSI DAN PENDAPATAN PER BULAN:
YearMonth  TotalTransactions  MonthlyRevenue
  2018-12               1552      4415